In [ ]:
# --- Bootstrap de Colab (no hace nada fuera de Colab) --------------------------------
# Colab arranca en /content con un entorno vacío: sin esta celda, el `import environment`
# siguiente falla con ModuleNotFoundError. La genera tools/py_to_notebook.py -- editala
# ahí, no acá.
import hashlib
import os
import subprocess
import sys

# El disco de Colab se borra cuando se desconecta el entorno de ejecución (~90 min inactivo,
# 12 h como máximo). Con USE_DRIVE = True, artifacts/ (documentos, embeddings, índices,
# resultados, cachés del LLM) vive en cambio en tu Google Drive, y al volver a correr se
# saltea todo lo que ya está construido.
USE_DRIVE = True  # Proyecto2_RAG: desde la 02, cada etapa lee artefactos de una etapa anterior
# Guardar también en Drive la caché de modelos de HuggingFace (~3 GB). Apagado por defecto:
# volver a descargar suele ser más rápido que leer archivos grandes a través del montaje de Drive.
MODELS_ON_DRIVE = False

# Un entorno de ejecución de Colab, desde el navegador o desde la extensión de Colab de VS Code
# (cuyo kernel puede no haber importado google.colab: la propia variable de entorno del runtime
# dice dónde estamos).
if "google.colab" in sys.modules or os.environ.get("COLAB_RELEASE_TAG"):
    REPO_URL = "https://github.com/facurie/Proyecto2_RAG.git"
    REPO_BRANCH = ""
    REPO_DIR = "Proyecto2_RAG"
    DRIVE = "/content/drive/MyDrive/Proyecto2_RAG"
    quiet = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}  # nunca esperar un pedido de contraseña

    def github_token():
        # Un repositorio privado: un token de GitHub como secreto de Colab llamado GITHUB_TOKEN.
        # Los secretos de Colab solo se pueden leer desde el navegador; una variable de entorno
        # funciona en cualquier lado.
        token = os.environ.get("GITHUB_TOKEN")
        if not token:
            try:
                from google.colab import userdata
                token = userdata.get("GITHUB_TOKEN")
            except Exception:
                token = None
        return token

    # Dónde ESTAMOS antes que qué hay en disco: `isdir(REPO_DIR)` es relativo, así que
    # chequearlo primero clonaría una copia anidada cada vez que se vuelve a correr esta celda
    # en la misma sesión.
    if os.path.basename(os.getcwd()) != REPO_DIR:
        if not os.path.isdir(REPO_DIR):
            cmd = ["git", "clone", "--depth", "1"] + (["--branch", REPO_BRANCH] if REPO_BRANCH else [])
            if subprocess.run(cmd + [REPO_URL, REPO_DIR], env=quiet).returncode != 0:
                token = github_token()
                if not token:
                    raise SystemExit("git clone failed. Is the repository public? For a private one, "
                                     "add a Colab secret named GITHUB_TOKEN (in browser Colab: VS Code's "
                                     "extension can't read Colab secrets).")
                authed = REPO_URL.replace("https://", "https://x-access-token:" + token + "@")
                # capture_output: un mensaje de error no tiene que imprimir el token en el notebook
                if subprocess.run(cmd + [authed, REPO_DIR], env=quiet, capture_output=True).returncode != 0:
                    raise SystemExit("git clone failed even with GITHUB_TOKEN: check that the token "
                                     "can read this repository (fine-grained: Contents: Read).")
                # El token NO se guarda en .git/config: un pull posterior lo vuelve a pasar (abajo).
                subprocess.run(["git", "-C", REPO_DIR, "remote", "set-url", "origin", REPO_URL])
        os.chdir(REPO_DIR)
    # Traer los arreglos publicados después de tu primer clon (no hace nada si no hay).
    pulled = subprocess.run(["git", "pull", "--ff-only", "-q"], capture_output=True, text=True, env=quiet)
    token = github_token() if pulled.returncode != 0 else None
    if token:   # un repositorio privado: volver a hacer pull con el token, sin guardarlo
        branch = subprocess.run(["git", "rev-parse", "--abbrev-ref", "HEAD"], capture_output=True,
                                text=True).stdout.strip()
        authed = REPO_URL.replace("https://", "https://x-access-token:" + token + "@")
        pulled = subprocess.run(["git", "pull", "--ff-only", "-q", authed, branch], capture_output=True,
                                text=True, env=quiet)
    if pulled.returncode != 0:
        message = (pulled.stderr or pulled.stdout).strip()
        print("note: git pull skipped:", (message.replace(token, "***") if token else message)[:200])
    if os.getcwd() not in sys.path:
        sys.path.insert(0, os.getcwd())

    # Instalar una vez por VM: una marca con clave en el contenido de los archivos de requirements.
    reqs = [f for f in ("requirements.txt", "requirements-group.txt") if os.path.isfile(f)]
    digest = hashlib.sha256(b"".join(open(f, "rb").read() for f in reqs)).hexdigest()[:12]
    stamp = "/content/.lab_pip_" + digest
    if not os.path.exists(stamp):
        pip = [sys.executable, "-m", "pip", "install", "-q"]
        for f in reqs:
            pip += ["-r", f]
        subprocess.run(pip, check=True)  # UNA sola resolución: una versión del grupo que choca con una del núcleo falla ruidosamente
        open(stamp, "w").close()

    if MODELS_ON_DRIVE and not USE_DRIVE:
        print("note: MODELS_ON_DRIVE needs USE_DRIVE = True; the model cache stays on the VM disk.")
    if USE_DRIVE:
        # ANTES de `import environment`: un directorio artifacts/ real creado primero le ganaría
        # en silencio al link, y todo terminaría en el disco efímero.
        if not os.path.isdir("/content/drive/MyDrive"):   # VS Code: ya montado por su comando
            # Se avisa ANTES de la llamada: en la extensión de Colab de VS Code, drive.mount() puede colgarse en vez de fallar.
            print("Mounting Drive... (in VS Code, if this hangs: interrupt, run 'Colab: Mount Google Drive "
                  "to Server...' from Ctrl+Shift+P, then re-run this cell)")
            try:
                from google.colab import drive
                drive.mount("/content/drive")
            except Exception as e:  # drive.mount() desde una celda no funciona en la extensión de Colab de VS Code
                raise SystemExit("Drive is not mounted (" + type(e).__name__ + "). In VS Code: Ctrl+Shift+P > "
                                 "'Colab: Mount Google Drive to Server...', then re-run this cell.")
        os.makedirs(DRIVE + "/artifacts", exist_ok=True)
        if os.path.islink("artifacts"):
            print("artifacts ->", os.readlink("artifacts"))
        elif os.path.isdir("artifacts"):
            print("WARNING: artifacts/ is already a real directory, so it is NOT on Drive.")
            print("Move its contents into", DRIVE + "/artifacts", ", delete it, then re-run this cell.")
        else:
            os.symlink(DRIVE + "/artifacts", "artifacts")
            print("artifacts ->", DRIVE + "/artifacts")
        os.environ["LAB_USE_DRIVE"] = "1"
        if MODELS_ON_DRIVE:
            os.makedirs(DRIVE + "/hf_cache", exist_ok=True)
            os.environ.setdefault("HF_HOME", DRIVE + "/hf_cache")

    print("Colab bootstrap OK. Working directory:", os.getcwd())

# Etapa 0 · Chequeo a libro cerrado

Cinco preguntas al generador (`qwen3:8b`), **sin recuperación**. Sirve para medir cuánto de nuestro corpus (legislación de vivienda y alquiler del BOE) ya sabe el modelo por su preentrenamiento. Si a libro cerrado responde la mitad, un eval set parecido a estas preguntas estaría midiendo memoria del modelo y no recuperación (assignment §7.4).

Las cinco preguntas cubren las categorías del eval set, con el gold verificado contra el texto descargado el 2026-10-16:

| # | Categoría | Qué pone a prueba |
|---|---|---|
| 1 | topical | un dato estable y muy difundido (prórroga obligatoria de la LAU) |
| 2 | metadata / temporal | la **versión** de un artículo vigente en una fecha (2016) |
| 3 | identifier | un artículo nombrado por número (art. 36 LAU, fianza) |
| 4 | topical (2023) | una definición de la Ley 12/2023 por el derecho a la vivienda |
| 5 | posterior al corte | el art. 9 bis LAU (vivienda temporal), que existe desde octubre de 2026 |

Qwen3 se publicó en abril de 2025, así que la pregunta 5 **no puede** saberla: es el control. Esperamos abstención (`can_answer=false`) o una alucinación, y cualquiera de las dos es información.

**Dónde corre.** En Colab con T4 (la corrida que cuenta para la entrega y para el presupuesto) o localmente con Ollama ya instalado. No usa el corpus ni `src/`. En Colab instala Ollama; localmente solo verifica que el servidor responda. Los **tiempos** solo sirven para el presupuesto si el modelo quedó 100% en la GPU de una T4: la notebook guarda el entorno, la GPU y la fracción en GPU junto con los resultados.

**Mismas reglas que el resto del proyecto:** API nativa (nunca `/v1`), `think=False`, temperatura 0, salida con schema, `num_ctx` explícito en cada llamada. Si algo falla (no levanta Ollama, en Colab el modelo no queda 100% en GPU, la salida no parsea), la notebook **lanza una excepción**: no avisa y sigue.

In [ ]:
# Variables de entorno ANTES de cualquier import pesado.
# JAX (si algo lo importa) preasigna el 75% de la VRAM; Ollama vería la placa llena y cargaría el
# modelo parcialmente en CPU, sin ningún error. Acá no usamos bm25s, pero lo dejamos por costumbre.
import os, subprocess
os.environ["OLLAMA_HOST"] = "127.0.0.1:11434"
os.environ["OLLAMA_NUM_PARALLEL"] = "1"
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["JAX_PLATFORMS"] = "cpu"

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
ENV_TAG = "colab" if IN_COLAB else "local"
try:
    GPU = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,memory.used", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=30).stdout.strip()
except FileNotFoundError:
    GPU = "sin GPU NVIDIA"
print("entorno:", ENV_TAG, "| GPU:", GPU)

In [ ]:
# Solo en Colab: instalar Ollama en la versión que fija el curso (llm.OLLAMA_VERSION), la misma de las
# Etapas 4+. Localmente se asume Ollama instalado por el usuario (en Windows, la app levanta el servidor).
if IN_COLAB:
    import llm  # del curso; también instala zstd, que el instalador necesita
    llm.install_ollama()

In [ ]:
import shutil, time, requests

HOST = "http://127.0.0.1:11434"

def ollama_alive() -> bool:
    try:
        return requests.get(f"{HOST}/api/tags", timeout=2).ok
    except requests.exceptions.RequestException:
        return False

if not ollama_alive():
    if IN_COLAB:
        # Un runner huérfano de un runtime anterior puede seguir reservando VRAM: matarlo antes.
        subprocess.run(["pkill", "-9", "-f", "ollama"], check=False)
        time.sleep(3)
    exe = shutil.which("ollama") or os.path.expandvars(r"%LOCALAPPDATA%\Programs\Ollama\ollama.exe")
    subprocess.Popen([exe, "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    for _ in range(60):
        if ollama_alive():
            break
        time.sleep(1)
if not ollama_alive():
    raise RuntimeError("Ollama no levantó en 60 s")
OLLAMA_VERSION = requests.get(f"{HOST}/api/version").json()["version"]
print("Ollama", OLLAMA_VERSION)

In [ ]:
# Tag explícito, nunca "qwen3" a secas. Guardamos además el digest: un tag se puede mover.
# El pull va por la API (no por el ejecutable): funciona igual en Colab y en Windows.
MODEL = "qwen3:8b"
r = requests.post(f"{HOST}/api/pull", json={"model": MODEL, "stream": False}, timeout=None)
if not r.ok or r.json().get("status") != "success":
    raise RuntimeError(f"pull falló: {r.status_code} {r.text[:300]}")
tags = {m["name"]: m for m in requests.get(f"{HOST}/api/tags").json()["models"]}
MODEL_DIGEST = tags[MODEL]["digest"]
print(MODEL, MODEL_DIGEST[:12], tags[MODEL]["details"])

## Las preguntas

Cada pregunta trae su gold (documento y bloque del BOE, y la versión que la responde), una `answer_key` corta y, si es de sí/no, su `expected_verdict`. El gold se verificó leyendo el texto descargado de la API (ver `decision_log.md`, Etapa 0).

Estas preguntas **no** entran al eval set: se redactaron con el pasaje a la vista, así que tienen el problema de contaminación que la Etapa 2 corrige.

In [ ]:
from pydantic import BaseModel, Field

class ClosedBookAnswer(BaseModel):
    # El razonamiento primero, y también primero en orden alfabético
    # (analysis < answer < can_answer): el modelo razona antes de comprometerse en cualquier servidor.
    analysis: str = Field(max_length=600)
    answer: str = Field(max_length=1200)
    can_answer: bool

QUESTIONS = [
    {"qid": "cb1", "category": "topical",
     "question": "Alquilé un piso a un particular para vivir en él y en el contrato pusimos un año de duración. Al terminar ese año, ¿el propietario me puede obligar a irme?",
     "gold": [("BOE-A-1994-26003", "a9", "version 2019-03-06 (RDL 7/2019)")],
     "answer_key": ["cinco años", "5 años"], "expected_verdict": "no"},
    {"qid": "cb2", "category": "metadata",
     "question": "Firmé un contrato de alquiler de vivienda habitual en septiembre de 2016. Según la ley vigente cuando lo firmé, ¿hasta cuántos años de duración mínima tenía garantizados?",
     "gold": [("BOE-A-1994-26003", "a9", "version 2013-06-06 (Ley 4/2013), vigente hasta 2018-12-19")],
     "answer_key": ["tres años", "3 años"], "expected_verdict": None},
    {"qid": "cb3", "category": "identifier",
     "question": "¿Qué fianza en metálico exige el artículo 36 de la Ley de Arrendamientos Urbanos cuando se alquila un local comercial?",
     "gold": [("BOE-A-1994-26003", "a36", "apartado 1, sin cambios desde 1995")],
     "answer_key": ["dos mensualidades", "dos meses", "2 mensualidades", "2 meses"], "expected_verdict": None},
    {"qid": "cb4", "category": "topical",
     "question": "Según la ley estatal por el derecho a la vivienda de 2023, ¿a partir de cuántas viviendas se considera que un propietario es un gran tenedor?",
     "gold": [("BOE-A-2023-12203", "a3", "letra k)")],
     "answer_key": ["diez", "10"], "expected_verdict": None},
    {"qid": "cb5", "category": "post_cutoff",
     "question": "Según la Ley de Arrendamientos Urbanos, ¿cuál es la duración máxima, como regla general, de un contrato de arrendamiento de vivienda temporal?",
     "gold": [("BOE-A-1994-26003", "a9-2", "art. 9 bis, version 2026-10-08 (RDL 29/2026)")],
     "answer_key": ["doce meses", "12 meses"], "expected_verdict": None},
]

SYSTEM = ("Sos un asistente que responde preguntas sobre la legislación española de vivienda y alquiler. "
          "Respondé en español, en pocas oraciones. Si la pregunta es de sí o no, empezá la respuesta con "
          "'Sí' o 'No'. Si no conocés la respuesta con seguridad, poné can_answer en false y decilo.")

In [ ]:
import json, re, unicodedata

NUM_CTX, NUM_PREDICT = 8192, 512   # fijos para las cinco llamadas

def ask(question: str) -> dict:
    payload = {
        "model": MODEL, "stream": False, "think": False,
        "messages": [{"role": "system", "content": SYSTEM}, {"role": "user", "content": question}],
        "format": ClosedBookAnswer.model_json_schema(),
        "options": {"temperature": 0, "seed": 0, "num_ctx": NUM_CTX, "num_predict": NUM_PREDICT},
    }
    r = requests.post(f"{HOST}/api/chat", json=payload, timeout=600)
    r.raise_for_status()
    out = r.json()
    content = out["message"]["content"]
    rec = {k: out.get(k) for k in ("done_reason", "prompt_eval_count", "eval_count",
                                   "total_duration", "prompt_eval_duration", "eval_duration", "load_duration")}
    rec["thinking_leak"] = "<think>" in content or "</think>" in content or bool(out["message"].get("thinking"))
    rec["raw"] = content
    # Un error de llamada es su propia categoría: ni abstención ni respuesta puntuada.
    if out.get("done_reason") != "stop":
        rec["error"] = f"done_reason={out.get('done_reason')}"
        return rec
    try:
        rec["parsed"] = ClosedBookAnswer.model_validate_json(content).model_dump()
    except Exception as e:
        rec["error"] = f"PARSE: {e}"
    return rec

def unload() -> None:
    # keep_alive=0 descarga el modelo: la próxima llamada arranca con la caché de prompts vacía.
    requests.post(f"{HOST}/api/generate", json={"model": MODEL, "keep_alive": 0}, timeout=120)
    time.sleep(3)

def placement() -> str:
    ps = requests.get(f"{HOST}/api/ps").json()["models"]
    m = next(m for m in ps if m["name"] == MODEL)
    return f"{100 * m['size_vram'] / m['size']:.0f}% GPU"

def norm(s: str) -> str:
    s = unicodedata.normalize("NFKD", s.lower())
    return "".join(c for c in s if not unicodedata.combining(c))

def key_in(key: str, text: str) -> bool:
    # Palabra o número completo, no substring: la clave "10" no puede contar "100 viviendas".
    return re.search(r"(?<!\w)" + re.escape(norm(key)) + r"(?!\w)", norm(text)) is not None

def verdict(answer: str) -> str:
    # Primera palabra, sobre el texto CRUDO: sacar acentos convierte "sí" en "si" (condicional).
    w = re.findall(r"\w+", answer)
    return {"sí": "yes", "no": "no"}.get(w[0].lower(), "unclear") if w else "unclear"

In [ ]:
# Arranque en frío: la salida depende del estado de la caché de prompts del servidor (ver más abajo).
unload()
results = []
for q in QUESTIONS:
    rec = ask(q["question"])
    rec.update(qid=q["qid"], category=q["category"])
    if "parsed" in rec:
        ans = rec["parsed"]["answer"]
        rec["key_hit"] = any(key_in(k, ans) for k in q["answer_key"])
        rec["verdict"] = verdict(ans) if q["expected_verdict"] else None
        # Una abstención no es un "no": "No lo sé" empieza con "no". El veredicto solo cuenta si respondió.
        rec["verdict_ok"] = (rec["parsed"]["can_answer"] and rec["verdict"] == q["expected_verdict"]) \
                            if q["expected_verdict"] else None
    results.append(rec)
    if len(results) == 1:
        where = placement()
        print("placement:", where)
        PLACEMENT = where
        if where != "100% GPU":
            if IN_COLAB:
                raise RuntimeError(f"El modelo no está entero en la GPU ({where}): los tiempos no sirven para el presupuesto")
            print(f"  local: el modelo quedó {where}; las respuestas valen, los tiempos NO son comparables con la T4")

# Determinismo: repetimos la primera pregunta. Si cambia con temperatura 0, eso es parte del piso de ruido.
rep = ask(QUESTIONS[0]["question"])
print("repetición idéntica:", rep.get("raw") == results[0].get("raw"))

In [ ]:
for r, q in zip(results, QUESTIONS):
    p = r.get("parsed", {})
    print(f"--- {r['qid']} [{r['category']}]  error={r.get('error')}  can_answer={p.get('can_answer')}  "
          f"key_hit={r.get('key_hit')}  verdict={r.get('verdict')} (esperado {q['expected_verdict']})  leak={r['thinking_leak']}")
    print("P:", q["question"])
    print("R:", p.get("answer"))
    print("clave:", q["answer_key"])

ok = [r for r in results if "error" not in r]
gen_s = [r["eval_duration"] / 1e9 for r in ok]
tok_s = [r["eval_count"] / (r["eval_duration"] / 1e9) for r in ok]
tot_s = [r["total_duration"] / 1e9 for r in ok[1:]]   # sin la primera: incluye la carga del modelo
print(f"\nclave contenida: {sum(r.get('key_hit', False) for r in ok)}/{len(QUESTIONS)}   "
      f"abstenciones: {sum(not r['parsed']['can_answer'] for r in ok)}   errores: {len(results) - len(ok)}")
print(f"tokens de salida/s: {min(tok_s):.1f}-{max(tok_s):.1f}   segundos por llamada (sin carga): "
      f"{min(tot_s):.1f}-{max(tot_s):.1f}   carga del modelo: {results[0]['load_duration'] / 1e9:.1f} s")

## Diagnóstico: ¿el resultado es del modelo o del formato?

Antes de leer el puntaje como "lo que sabe el modelo", descartamos que lo haya producido nuestro prompt o el schema. La primera corrida local mostró dos señales sospechosas: el modelo empezaba **todas** las respuestas con "Sí" (la instrucción de sí/no del prompt se aplicaba también a preguntas que no lo son) y, al abstenerse, escribía en `answer` el nombre de la clave siguiente ("can_answer"). Repetimos las cinco preguntas con dos variantes:

- **schema sin la instrucción de sí/no**;
- **texto libre**, sin schema.

Si las variantes aciertan donde la original falla, el puntaje era del formato. La contención acá se mide sobre todo el texto devuelto (incluido `analysis`): es un diagnóstico, no un puntaje.

In [ ]:
SYS_NEUTRAL = ("Sos un asistente que responde preguntas sobre la legislación española de vivienda y alquiler. "
               "Respondé en español, en pocas oraciones. Si no conocés la respuesta con seguridad, decilo.")

def chat_raw(system: str, question: str, fmt) -> str:
    payload = {"model": MODEL, "stream": False, "think": False,
               "messages": [{"role": "system", "content": system}, {"role": "user", "content": question}],
               "options": {"temperature": 0, "seed": 0, "num_ctx": NUM_CTX, "num_predict": NUM_PREDICT}}
    if fmt is not None:
        payload["format"] = fmt
    r = requests.post(f"{HOST}/api/chat", json=payload, timeout=600)
    r.raise_for_status()
    return r.json()["message"]["content"]

diag = []
for q in QUESTIONS:
    a = chat_raw(SYS_NEUTRAL + " Si no podés responder, poné can_answer en false.", q["question"],
                 ClosedBookAnswer.model_json_schema())
    b = chat_raw(SYS_NEUTRAL, q["question"], None)
    diag.append({"qid": q["qid"], "schema_sin_si_no": a, "texto_libre": b,
                 "key_hit_schema": any(key_in(k, a) for k in q["answer_key"]),
                 "key_hit_libre": any(key_in(k, b) for k in q["answer_key"])})
    print(f"--- {q['qid']}  clave={q['answer_key']}")
    print("  schema sin sí/no:", a[:400])
    print("  texto libre:     ", b[:400])
print(f"\nclave contenida: original {sum(r.get('key_hit', False) for r in results)}/5 · "
      f"schema sin sí/no {sum(d['key_hit_schema'] for d in diag)}/5 · texto libre {sum(d['key_hit_libre'] for d in diag)}/5")

## ¿Temperatura 0 alcanza para que sea determinista?

No. En la corrida local, la pregunta 2 respondió "1 año" en una corrida y "3 años" en la siguiente, con el mismo prompt, temperatura 0 y `seed` 0. El log del servidor mostró la diferencia: en un caso reutilizó de su **caché de prompts** el prefijo que la pregunta 2 comparte con la 1 (el prompt de sistema) y calculó el resto en otro lote; en el otro tenía el prompt entero cacheado. La caché KV calculada en lotes distintos difiere levemente en punto flotante, y cuando dos tokens están casi empatados la decodificación greedy puede caer distinto.

El experimento controlado: con el modelo recién cargado, (A) preguntar la 1 y después la 2, contra (B) la 2 sola. Si A y B difieren, **el orden de las llamadas cambia las respuestas**, y eso entra en el piso de ruido de cada comparación entre brazos (assignment §7.4). La repetición inmediata de arriba no lo detecta: repite con la caché en el mismo estado. Por eso las cinco preguntas principales arrancan con el modelo descargado: así cada corrida empieza desde el mismo estado.

In [ ]:
cache_exp = []
for trial in range(2):
    unload(); ask(QUESTIONS[0]["question"]); a = ask(QUESTIONS[1]["question"])
    unload(); b = ask(QUESTIONS[1]["question"])
    ans_a, ans_b = a.get("parsed", {}).get("answer"), b.get("parsed", {}).get("answer")
    cache_exp.append({"trial": trial + 1, "A_after_cb1": ans_a, "B_cold": ans_b, "same": ans_a == ans_b})
    print(f"intento {trial + 1}: A (después de la 1) = {ans_a!r}")
    print(f"           B (sola, en frío)  = {ans_b!r}")
print("\n¿el orden cambia la respuesta?", any(not e["same"] for e in cache_exp))

In [ ]:
# Guardar las salidas crudas con su configuración. En Colab, a Drive si está montado.
import pathlib, datetime
if IN_COLAB and pathlib.Path("/content/drive").exists():
    out_dir = pathlib.Path("/content/drive/MyDrive/Proyecto2_RAG/artifacts/full")
else:
    root = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
    out_dir = root / "artifacts" / "full"
out_dir.mkdir(parents=True, exist_ok=True)
run = {"stage": 0, "what": "closed_book_spot_check", "run_at": datetime.datetime.now().isoformat(timespec="seconds"),
       "env": ENV_TAG, "gpu": GPU, "placement": PLACEMENT,
       "model": MODEL, "model_digest": MODEL_DIGEST, "ollama_version": OLLAMA_VERSION,
       "num_ctx": NUM_CTX, "num_predict": NUM_PREDICT, "system": SYSTEM,
       "questions": QUESTIONS, "results": results, "repeat_identical": rep.get("raw") == results[0].get("raw"), "diagnostic": diag, "cache_order_experiment": cache_exp}
path = out_dir / f"closed_book_stage0_{ENV_TAG}.json"
path.write_text(json.dumps(run, ensure_ascii=False, indent=1), encoding="utf-8")
print("guardado en", path)

## Cómo leer el resultado

- **Leé las cinco respuestas enteras**, no solo `key_hit`: la contención no distingue un "5 años" correcto de un "5 años" dicho para la versión equivocada (pregunta 2).
- **Si acierta 3 o más de 1-4**, el corpus vigente es en buena parte "conocido" por el modelo. Entonces el eval set tiene que apoyarse en lo que el modelo no puede saber: versiones fechadas, cambios de 2025-2026, normas autonómicas y detalles de artículos poco citados. La fila de libro cerrado de la Etapa 4 lo va a medir sobre el set completo.
- **Pregunta 5:** abstenerse es lo correcto. Si inventa una duración con seguridad, es una buena evidencia para el informe de por qué hace falta recuperación, y de por qué la abstención se mide con el booleano.
- Los tiempos por llamada alimentan el presupuesto de `decision_log.md`: reemplazan la estimación de 10-20 s por una medición en nuestra T4.

Volcá los números (aciertos, abstenciones, tokens/s, s/llamada) en `decision_log.md`.